In [1]:
from enum import Enum

In [2]:
# Enums (mirror models.py exactly)
class SeverityLabel:
    non_serious      = "non_serious"
    serious          = "serious"
    life_threatening = "life_threatening"
    fatal            = "fatal"

class RecommendedAction:
    monitor_only                   = "monitor_only"
    request_followup               = "request_followup"
    expedited_review               = "expedited_review"
    signal_team_review             = "signal_team_review"
    urgent_regulatory_notification = "urgent_regulatory_notification"

In [3]:
# Task 1 Grader
def grade_task1(action: dict, ground_truth: dict) -> dict:
    correct = action["severity_label"] == ground_truth["gt_severity"]
    score = 1.0 if correct else 0.0
    return {
        "total_score":     score,
        "severity_correct": correct,
        "task_id":         "task1"
    }


# Task 2 Grader
def grade_task2(action: dict, ground_truth: dict) -> dict:
    sev_score = 1.0 if action["severity_label"] == ground_truth["gt_severity"] else 0.0
    cau_score = round(1.0 - abs(action["causality_score"] - ground_truth["gt_causality"]), 4)
    esc_score = 1.0 if action["escalate_flag"] == ground_truth["gt_escalate"] else 0.0
    act_score = 1.0 if action["recommended_action"] == ground_truth["gt_recommended_action"] else 0.0

    total = round(
        0.4 * sev_score +
        0.3 * cau_score +
        0.2 * esc_score +
        0.1 * act_score,
        4
    )

    return {
        "total_score":      total,
        "severity_score":   sev_score,
        "causality_score":  cau_score,
        "escalation_score": esc_score,
        "action_score":     act_score,
        "task_id":          "task2"
    }


# Task 3 Grader
def grade_task3(action: dict, ground_truth: dict) -> dict:
    pred   = action["escalate_flag"]
    actual = ground_truth["gt_escalate"]

    tp = pred and actual
    fp = pred and not actual
    fn = not pred and actual

    precision = 1.0 if tp else (0.0 if fp else 1.0)
    recall    = 1.0 if tp else (0.0 if fn else 1.0)
    f1        = round((2 * precision * recall) / (precision + recall + 1e-8), 4)

    action_bonus = 0.1 if action["recommended_action"] == ground_truth["gt_recommended_action"] else 0.0
    total = round(min(f1 + action_bonus, 1.0), 4)

    return {
        "total_score":  total,
        "f1_score":     f1,
        "action_bonus": action_bonus,
        "precision":    precision,
        "recall":       recall,
        "task_id":      "task3"
    }


In [4]:
print("Graders defined.")

Graders defined.


In [5]:
# ---- Tests ----

# Task 1 tests
print(" TASK 1 TESTS ")
# Perfect score
r = grade_task1(
    {"severity_label": "serious"},
    {"gt_severity": "serious"}
)
print("All correct:", r)

# Wrong answer
r = grade_task1(
    {"severity_label": "non_serious"},
    {"gt_severity": "serious"}
)
print("All wrong:", r)

# Task 2 tests
print("\n TASK 2 TESTS ")
# Perfect score
r = grade_task2(
    {"severity_label": "serious", "causality_score": 0.8, "escalate_flag": True, "recommended_action": "request_followup"},
    {"gt_severity": "serious", "gt_causality": 0.8, "gt_escalate": True, "gt_recommended_action": "request_followup"}
)
print("All correct:", r)

# All wrong
r = grade_task2(
    {"severity_label": "non_serious", "causality_score": 0.1, "escalate_flag": False, "recommended_action": "monitor_only"},
    {"gt_severity": "serious", "gt_causality": 0.8, "gt_escalate": True, "gt_recommended_action": "request_followup"}
)
print("All wrong:", r)

# Partial credit
r = grade_task2(
    {"severity_label": "serious", "causality_score": 0.5, "escalate_flag": True, "recommended_action": "monitor_only"},
    {"gt_severity": "serious", "gt_causality": 0.8, "gt_escalate": True, "gt_recommended_action": "request_followup"}
)
print("Partial:", r)

# Task 3 tests
print("\n TASK 3 TESTS ")
# True positive — correctly identified signal
r = grade_task3(
    {"escalate_flag": True,  "recommended_action": "signal_team_review"},
    {"gt_escalate":   True,  "gt_recommended_action": "signal_team_review"}
)
print("True positive + correct action:", r)

# False positive — said signal when not
r = grade_task3(
    {"escalate_flag": True,  "recommended_action": "signal_team_review"},
    {"gt_escalate":   False, "gt_recommended_action": "monitor_only"}
)
print("False positive:", r)

# False negative — missed real signal
r = grade_task3(
    {"escalate_flag": False, "recommended_action": "monitor_only"},
    {"gt_escalate":   True,  "gt_recommended_action": "signal_team_review"}
)
print("False negative:", r)

# True negative — correctly said no signal
r = grade_task3(
    {"escalate_flag": False, "recommended_action": "monitor_only"},
    {"gt_escalate":   False, "gt_recommended_action": "monitor_only"}
)
print("True negative + correct action:", r)

 TASK 1 TESTS 
All correct: {'total_score': 1.0, 'severity_correct': True, 'task_id': 'task1'}
All wrong: {'total_score': 0.0, 'severity_correct': False, 'task_id': 'task1'}

 TASK 2 TESTS 
All correct: {'total_score': 1.0, 'severity_score': 1.0, 'causality_score': 1.0, 'escalation_score': 1.0, 'action_score': 1.0, 'task_id': 'task2'}
All wrong: {'total_score': 0.09, 'severity_score': 0.0, 'causality_score': 0.3, 'escalation_score': 0.0, 'action_score': 0.0, 'task_id': 'task2'}
Partial: {'total_score': 0.81, 'severity_score': 1.0, 'causality_score': 0.7, 'escalation_score': 1.0, 'action_score': 0.0, 'task_id': 'task2'}

 TASK 3 TESTS 
True positive + correct action: {'total_score': 1.0, 'f1_score': 1.0, 'action_bonus': 0.1, 'precision': 1.0, 'recall': 1.0, 'task_id': 'task3'}
False positive: {'total_score': 0.0, 'f1_score': 0.0, 'action_bonus': 0.0, 'precision': 0.0, 'recall': 1.0, 'task_id': 'task3'}
False negative: {'total_score': 0.0, 'f1_score': 0.0, 'action_bonus': 0.0, 'precision

In [6]:
import json

with open("../data/processed/task1_dataset.json") as f:
    t1 = json.load(f)
with open("../data/processed/task2_dataset.json") as f:
    t2 = json.load(f)
with open("../data/processed/task3_dataset.json") as f:
    t3 = json.load(f)

# Task 1 — always predict serious, check score variance
scores1 = [
    grade_task1({"severity_label": "serious"}, r)["total_score"]
    for r in t1
]
print(f"Task 1 — unique scores: {set(scores1)}")

# Task 2 — always predict same thing, check score variance
scores2 = [
    grade_task2(
        {"severity_label": "serious", "causality_score": 0.8,
         "escalate_flag": True, "recommended_action": "request_followup"},
        r
    )["total_score"]
    for r in t2
]
print(f"Task 2 — unique scores: {set(scores2)}")

# Task 3 — always escalate, check score variance
scores3 = [
    grade_task3(
        {"escalate_flag": True, "recommended_action": "signal_team_review"},
        c
    )["total_score"]
    for c in t3
]
print(f"Task 3 — unique scores: {set(scores3)}")

Task 1 — unique scores: {0.0, 1.0}
Task 2 — unique scores: {0.91, 1.0, 0.21, 0.3, 0.5}
Task 3 — unique scores: {0.0, 1.0}
